# DK AIS [`dkais`] [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andyrids/dkais/blob/main/docs/examples/DKAIS_EXAMPLE.ipynb)

This Notebook demonstrates the use of `dkais` CLI commands and functionality with practical examples.

**NOTE**: When using the `process` command, a day's CSV is held in memory ONLY when five times its size is available (a 3 GB daily CSV needs ~15 GB), otherwise it is extracted to `<data-dir>/tmp`. Set the Notebook runtime to 'High RAM' for quicker processing.

In [1]:
!uv pip install dkais keplergl polars[pandas]

Using Python 3.13.16 environment at: /usr
Checked 3 packages in 91ms


In [2]:
import pathlib

import polars as pl
from google.colab import drive
from IPython.display import clear_output
from keplergl import KeplerGl
from rich.pretty import pprint

from dkais.logging import GLOBAL_CONSOLE

If Google Drive is not connected, then the data will be stored in the ephemeral storage.

In [3]:
drive.mount("/content/drive", force_remount=True)
clear_output(wait=True)

Mounted at /content/drive


Use the `list` command to display the remote files between `--start` (2026-10-15) and `--end` (2026-10-20). The `--end` can be omitted to list upto today.

The `Raw` column is unchecked and `Processed` is 0/1 as we set the data
directory, allowing the CLI to check existing raw CSV zip and
processed Parquet files agaisnt the remote listing.

**NOTE**: In these example cells, the data directory is being indicated with
the DKAIS_DATA_DIR environment variable. Passing the argument
`--data-dir="/content/drive/MyDrive/DATA/AISDK/"` would also work.

In [4]:
!env DKAIS_DATA_DIR="/content/drive/MyDrive/DATA/AISDK/" \
    uv run dkais list --start 2026-10-15 --end 2026-10-20

                                                                             
 Key                    Start        End              Size   Raw   Processed 
 ─────────────────────────────────────────────────────────────────────────── 
 aisdk-2026-10-15.zip   2026-10-15   2026-10-15   629.8 MB    ✓          1/1 
 aisdk-2026-10-16.zip   2026-10-16   2026-10-16   637.8 MB    ✓          1/1 
 aisdk-2026-10-17.zip   2026-10-17   2026-10-17   599.7 MB    ✓          1/1 
 aisdk-2026-10-18.zip   2026-10-18   2026-10-18   612.3 MB    ✓          1/1 
 aisdk-2026-10-19.zip   2026-10-19   2026-10-19   606.2 MB    ✓          1/1 
 aisdk-2026-10-20.zip   2026-10-20   2026-10-20   595.8 MB    ✓          1/1 
                                                                             
                             6 archives, 3.7 GB                              


Use the `download` command to download the files between the same dates as above.

In [5]:
!env DKAIS_DATA_DIR="/content/drive/MyDrive/DATA/AISDK/" \
    uv run dkais download --start 2026-10-15 --end 2026-10-20

0 archives to download · 0 bytes · 6 already present
⠋ ━━━━━━━━━━━━━━━━━━━━ 0/0 bytes ?                                              
✓ Downloaded 0 archives (0 bytes) in 0.0s, 6 already present →
  /content/drive/MyDrive/DATA/AISDK/raw


Trying to re-download the same files, will display an info message rather
than download them again, unless the `--force` argument is passed.

In [6]:
!env DKAIS_DATA_DIR="/content/drive/MyDrive/DATA/AISDK/" \
    uv run dkais download --start 2026-10-15 --end 2026-10-20

0 archives to download · 0 bytes · 6 already present
⠋ ━━━━━━━━━━━━━━━━━━━━ 0/0 bytes ?                                              
✓ Downloaded 0 archives (0 bytes) in 0.0s, 6 already present →
  /content/drive/MyDrive/DATA/AISDK/raw


Listing the same remote files again, shows that the raw CSV zip files for this time period have been downloaded into the data directory - denoted by a
tick in the `Raw` column.

In [19]:
!env DKAIS_DATA_DIR="/content/drive/MyDrive/DATA/AISDK/" \
    uv run dkais list --start 2026-10-15 --end 2026-10-20

                                                                             
 Key                    Start        End              Size   Raw   Processed 
 ─────────────────────────────────────────────────────────────────────────── 
 aisdk-2026-10-15.zip   2026-10-15   2026-10-15   629.8 MB    ✓          0/1 
 aisdk-2026-10-16.zip   2026-10-16   2026-10-16   637.8 MB    ✓          0/1 
 aisdk-2026-10-17.zip   2026-10-17   2026-10-17   599.7 MB    ✓          0/1 
 aisdk-2026-10-18.zip   2026-10-18   2026-10-18   612.3 MB    ✓          0/1 
 aisdk-2026-10-19.zip   2026-10-19   2026-10-19   606.2 MB    ✓          0/1 
 aisdk-2026-10-20.zip   2026-10-20   2026-10-20   595.8 MB    ✓          0/1 
                                                                             
                             6 archives, 3.7 GB                              


Next, we run the `process` command to transform the RAW CSV zip files into
partitioned Parquet.

In [8]:
!env DKAIS_DATA_DIR="/content/drive/MyDrive/DATA/AISDK/" \
    uv run dkais process --start 2026-10-15 --end 2026-10-20

6 days to process · 18.2 GB CSV · 0 already processed
  ━━━━━━━━━━━━━━━━━━━━ 6/6                                                      
✓ Processed 6 days in 2m11s → /content/drive/MyDrive/DATA/AISDK/parquet


Trying to process already transfomred files, will display a warning message,
unless the `--force` argument is passed, re-proecessing each CSV zip in the
designated period.

In [7]:
!env DKAIS_DATA_DIR="/content/drive/MyDrive/DATA/AISDK/" \
    uv run dkais process --start 2026-10-15 --end 2026-10-20

The `list` command now shows that all data from this period has been downloaded and processed.

In [ ]:
!env DKAIS_DATA_DIR="/content/drive/MyDrive/DATA/AISDK/" \
    uv run dkais list --start 2026-10-15 --end 2026-10-20

                                                                             
 Key                    Start        End              Size   Raw   Processed 
 ─────────────────────────────────────────────────────────────────────────── 
 aisdk-2026-10-15.zip   2026-10-15   2026-10-15   629.8 MB    ✓          1/1 
 aisdk-2026-10-16.zip   2026-10-16   2026-10-16   637.8 MB    ✓          1/1 
 aisdk-2026-10-17.zip   2026-10-17   2026-10-17   599.7 MB    ✓          1/1 
 aisdk-2026-10-18.zip   2026-10-18   2026-10-18   612.3 MB    ✓          1/1 
 aisdk-2026-10-19.zip   2026-10-19   2026-10-19   606.2 MB    ✓          1/1 
 aisdk-2026-10-20.zip   2026-10-20   2026-10-20   595.8 MB    ✓          1/1 
                                                                             
                             6 archives, 3.7 GB                              


## Exploring the data

These next cells demonstrate working with the Parquet files in Polars.

In [4]:
DATA_DIR = "/content/drive/MyDrive/DATA/AISDK/"
PARQUET_DIR = pathlib.Path(DATA_DIR) / "parquet"

In [5]:
ais = pl.scan_parquet(PARQUET_DIR, hive_partitioning=True)
october = ais.filter(pl.col("YEAR") == 2026, pl.col("MONTH") == 10)

In [6]:
pprint(october.collect_schema().to_python(), expand_all=True)

{
│   'DATETIME': <class 'datetime.datetime'>,
│   'MOBILE_TYPE': <class 'str'>,
│   'MMSI': <class 'int'>,
│   'LATITUDE': <class 'float'>,
│   'LONGITUDE': <class 'float'>,
│   'NAV_STATUS': <class 'str'>,
│   'ROT': <class 'float'>,
│   'SOG': <class 'float'>,
│   'COG': <class 'float'>,
│   'HEADING': <class 'int'>,
│   'IMO': <class 'int'>,
│   'CALLSIGN': <class 'str'>,
│   'NAME': <class 'str'>,
│   'SHIP_TYPE': <class 'str'>,
│   'CARGO_TYPE': <class 'str'>,
│   'WIDTH': <class 'int'>,
│   'LENGTH': <class 'int'>,
│   'POSITION_DEVICE': <class 'str'>,
│   'DRAUGHT': <class 'float'>,
│   'DESTINATION': <class 'str'>,
│   'ETA': <class 'datetime.datetime'>,
│   'DATA_SOURCE': <class 'str'>,
│   'SIZE_A': <class 'int'>,
│   'SIZE_B': <class 'int'>,
│   'SIZE_C': <class 'int'>,
│   'SIZE_D': <class 'int'>,
│   'YEAR': <class 'int'>,
│   'MONTH': <class 'int'>
}

In [ ]:
count = october.select(pl.len()).collect().item()
GLOBAL_CONSOLE.print(
    f"[b]COUNT[/b]: [orange3]{count:,}[/orange3]", markup=True
)

COUNT: 100,724,657

In [8]:
mobile_types = october.select("MOBILE_TYPE").unique().drop_nulls().collect()
display(mobile_types)

MOBILE_TYPE
cat
"""Class A"""
"""Class B"""
"""Base Station"""
"""SAR Airborne"""
"""Man Overboard Device"""
"""Emergency PIRB"""
"""AtoN"""
"""Search and Rescue Transponder"""


In [9]:
# Aggregate information per MMSI
mmsi_summary = (
    october.filter(pl.col("MOBILE_TYPE").is_in(["Class A", "Class B"]))
    .group_by("MMSI")
    .agg(
        NAME=pl.col("NAME").drop_nulls().unique(),
        RECORDS=pl.len(),
        FIRST_SEEN=pl.col("DATETIME").min(),
        LAST_SEEN=pl.col("DATETIME").max(),
        MOBILE_TYPE=pl.col("MOBILE_TYPE").drop_nulls().first(),
        SHIP_TYPE=pl.col("SHIP_TYPE").drop_nulls().first(),
    )
    .sort("RECORDS", descending=True)
)

# Collect and display the top 10 results
display(mmsi_summary.head(10).collect())

MMSI,NAME,RECORDS,FIRST_SEEN,LAST_SEEN,MOBILE_TYPE,SHIP_TYPE
i32,list[str],u32,datetime[μs],datetime[μs],cat,cat
3638,"[""3638""]",264094,2026-10-15 00:00:05,2026-10-20 23:59:50,"""Class A""","""Undefined"""
219022903,"[""EXPRESS 3""]",244220,2026-10-15 00:00:19,2026-10-20 23:59:56,"""Class A""","""Undefined"""
219005068,"[""M.F. OMOE""]",237732,2026-10-15 00:00:07,2026-10-20 23:59:58,"""Class A""","""Undefined"""
219023834,"[""DANPILOT ECHO""]",231918,2026-10-15 00:00:01,2026-10-20 23:59:54,"""Class A""","""Undefined"""
211188000,"[""DEUTSCHLAND""]",210052,2026-10-15 00:00:05,2026-10-20 23:59:50,"""Class A""","""Undefined"""
245693000,"[""KILSTROOM""]",208820,2026-10-15 00:00:00,2026-10-20 23:59:49,"""Class A""","""Undefined"""
219018172,"[""EXPRESS 2""]",204717,2026-10-15 00:00:05,2026-10-20 23:59:50,"""Class A""","""Undefined"""
219034268,"[""WIND DISCOVERY""]",200227,2026-10-15 00:00:03,2026-10-20 23:58:29,"""Class A""","""Undefined"""
219705000,"[""EXPRESS 4""]",196196,2026-10-15 00:00:09,2026-10-20 23:59:52,"""Class A""","""Undefined"""


In [ ]:
# Filter for Class A/B vessels
october_ab = october.filter(
    pl.col("MOBILE_TYPE").is_in(["Class A", "Class B"])
)

# Check the row count after filtering
count_ab = october_ab.select(pl.len()).collect().item()
GLOBAL_CONSOLE.print(
    f"[b]COUNT (Class A & B)[/b]: [orange3]{count_ab:,}[/orange3]",
    markup=True,
    end="\n\n",
)

# Display a few rows to verify
display(october_ab.head().collect())

COUNT (Class A & B): 92,544,204

DATETIME,MOBILE_TYPE,MMSI,LATITUDE,LONGITUDE,NAV_STATUS,ROT,SOG,COG,HEADING,IMO,CALLSIGN,NAME,SHIP_TYPE,CARGO_TYPE,WIDTH,LENGTH,POSITION_DEVICE,DRAUGHT,DESTINATION,ETA,DATA_SOURCE,SIZE_A,SIZE_B,SIZE_C,SIZE_D,YEAR,MONTH
datetime[μs],cat,i32,f64,f64,cat,f32,f32,f32,i16,i32,str,str,cat,cat,i16,i16,cat,f32,str,datetime[μs],cat,i16,i16,i16,i16,i64,i64
2026-10-15 05:35:25,"""Class A""",1111,56.217738,3.421677,"""Unknown value""",null,72.0,253.0,null,null,"""R4A""","""TEST""","""Undefined""",null,null,null,"""Internal""",null,null,null,"""AIS""",null,null,null,null,2026,10
2026-10-15 16:31:24,"""Class A""",1111,56.426148,3.261108,"""Unknown value""",null,140.0,150.300003,null,null,"""R4A""","""TEST""","""Undefined""",null,null,null,"""Internal""",null,null,null,"""AIS""",null,null,null,null,2026,10
2026-10-15 16:43:25,"""Class A""",1111,56.24041,3.420065,"""Unknown value""",null,109.0,3.7,null,null,"""R4A""","""TEST""","""Undefined""",null,null,null,"""Internal""",null,null,null,"""AIS""",null,null,null,null,2026,10
2026-10-15 00:00:05,"""Class A""",3638,55.332243,10.964893,"""Unknown value""",null,0.0,0.0,null,null,null,null,"""Undefined""",null,null,null,"""Undefined""",null,null,null,"""AIS""",null,null,null,null,2026,10
2026-10-15 00:00:05,"""Class A""",3638,55.332243,10.964893,"""Unknown value""",null,0.0,0.0,null,null,null,null,"""Undefined""",null,null,null,"""Undefined""",null,null,null,"""AIS""",null,null,null,null,2026,10


In [ ]:
kmap = KeplerGl()

kmap.add_data(
    october_ab.filter(pl.col("MMSI").eq(219005068)).collect().to_pandas(),
    name="MMSI",
)

kmap